# Challenge Lab 3 — Multi-Agent System

In [1]:
!pip install google-adk requests

## Configure Vertex AI

In [2]:
import os

project_id = !gcloud config get-value project
PROJECT_ID = project_id[0].strip()
print(f"Project ID: {PROJECT_ID}")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_MAPS_API_KEY"] = "{GOOGLE_MAPS_API}"

print("Vertex AI backend configured")

Project ID: qwiklabs-gcp-00-93a45da1459f
Vertex AI backend configured


## Create project structure

In [3]:
!mkdir -p multi_agent

In [4]:
%%writefile multi_agent/__init__.py


Overwriting multi_agent/__init__.py


In [5]:
%%bash
cat > multi_agent/.env << EOF
GOOGLE_CLOUD_PROJECT=$(gcloud config get-value project)
GOOGLE_CLOUD_LOCATION=us-central1
GOOGLE_GENAI_USE_VERTEXAI=TRUE
GOOGLE_MAPS_API_KEY={GOOGLE_MAPS_API}
EOF
cat multi_agent/.env

GOOGLE_CLOUD_PROJECT=qwiklabs-gcp-00-93a45da1459f
GOOGLE_CLOUD_LOCATION=us-central1
GOOGLE_GENAI_USE_VERTEXAI=TRUE
GOOGLE_MAPS_API_KEY={GOOGLE_MAPS_API}


## Agent module — weather_agent + search_agent + root_agent

In [24]:
%%writefile multi_agent/agent.py
import os
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from google.adk.agents import Agent
from google.adk.tools import google_search, agent_tool

# ── NWS session with retries ────────────────────────────────────────────────

NWS_HEADERS = {"User-Agent": "(multi_agent, contact@example.com)"}
NWS_TIMEOUT = 30

nws_session = requests.Session()
nws_session.headers.update(NWS_HEADERS)
retry_strategy = Retry(total=3, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
nws_session.mount("https://", HTTPAdapter(max_retries=retry_strategy))


# ── Weather tools ────────────────────────────────────────────────────────────

def get_coordinates(city: str) -> dict:
    """Convert a city name to latitude and longitude using Google Maps Geocoding API."""
    api_key = os.getenv("GOOGLE_MAPS_API_KEY")
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    resp = requests.get(url, params={"address": city, "key": api_key}, timeout=10)
    data = resp.json()
    if data.get("status") != "OK" or not data.get("results"):
        return {"error": f"Could not geocode '{city}'. Status: {data.get('status')}"}
    location = data["results"][0]["geometry"]["location"]
    formatted = data["results"][0].get("formatted_address", city)
    return {
        "latitude": location["lat"],
        "longitude": location["lng"],
        "formatted_address": formatted,
    }


def get_weather(latitude: float, longitude: float) -> dict:
    """Get the weather forecast for a location using the National Weather Service API (US only)."""
    try:
        points_url = f"https://api.weather.gov/points/{latitude},{longitude}"
        points_resp = nws_session.get(points_url, timeout=NWS_TIMEOUT)
        if points_resp.status_code != 200:
            return {"error": f"NWS points lookup failed (status {points_resp.status_code}). US locations only."}
        forecast_url = points_resp.json()["properties"]["forecast"]
        forecast_resp = nws_session.get(forecast_url, timeout=NWS_TIMEOUT)
        if forecast_resp.status_code != 200:
            return {"error": f"NWS forecast fetch failed (status {forecast_resp.status_code})."}
        periods = forecast_resp.json()["properties"]["periods"]
        return {
            "forecast": [
                {
                    "name": p["name"],
                    "temperature": p["temperature"],
                    "temperatureUnit": p["temperatureUnit"],
                    "windSpeed": p["windSpeed"],
                    "windDirection": p["windDirection"],
                    "shortForecast": p["shortForecast"],
                    "detailedForecast": p["detailedForecast"],
                }
                for p in periods[:6]
            ]
        }
    except requests.exceptions.Timeout:
        return {"error": "NWS API timed out. Try again."}


def get_current_conditions(latitude: float, longitude: float) -> dict:
    """Get current weather conditions from the nearest observation station (US only)."""
    try:
        points_url = f"https://api.weather.gov/points/{latitude},{longitude}"
        points_resp = nws_session.get(points_url, timeout=NWS_TIMEOUT)
        if points_resp.status_code != 200:
            return {"error": f"NWS points lookup failed (status {points_resp.status_code}). US locations only."}
        stations_url = points_resp.json()["properties"]["observationStations"]
        stations_resp = nws_session.get(stations_url, timeout=NWS_TIMEOUT)
        if stations_resp.status_code != 200:
            return {"error": f"NWS stations fetch failed (status {stations_resp.status_code})."}
        station_id = stations_resp.json()["features"][0]["properties"]["stationIdentifier"]
        obs_url = f"https://api.weather.gov/stations/{station_id}/observations/latest"
        obs_resp = nws_session.get(obs_url, timeout=NWS_TIMEOUT)
        if obs_resp.status_code != 200:
            return {"error": f"NWS observation fetch failed (status {obs_resp.status_code})."}
        props = obs_resp.json()["properties"]
        temp_c = props.get("temperature", {}).get("value")
        temp_f = round(temp_c * 9 / 5 + 32, 1) if temp_c is not None else None
        wind_ms = props.get("windSpeed", {}).get("value")
        wind_mph = round(wind_ms * 2.237, 1) if wind_ms is not None else None
        return {
            "station": station_id,
            "description": props.get("textDescription", ""),
            "temperature_f": temp_f,
            "temperature_c": round(temp_c, 1) if temp_c is not None else None,
            "humidity": props.get("relativeHumidity", {}).get("value"),
            "wind_speed_mph": wind_mph,
            "wind_direction_degrees": props.get("windDirection", {}).get("value"),
        }
    except requests.exceptions.Timeout:
        return {"error": "NWS API timed out. Try again."}


# ── Sub-agent: Weather ───────────────────────────────────────────────────────

weather_agent = Agent(
    name="weather_agent",
    model="gemini-2.5-flash",
    description="Handles weather queries for US cities. Use this agent for any weather, forecast, or current conditions requests.",
    instruction=(
        "You are a weather specialist. When asked about weather for a city, "
        "use get_coordinates to find its lat/lon, then use get_weather for forecasts "
        "or get_current_conditions for current observations. "
        "The NWS API only covers US locations — tell the user if they ask about non-US cities. "
        "If the user asks for just a high or low, still fetch the full forecast and extract the relevant value."
    ),
    tools=[get_coordinates, get_weather, get_current_conditions],
)


# ── Search agent (wrapped as AgentTool — google_search can't mix with other tool types) ─

search_agent = Agent(
    name="search_agent",
    model="gemini-2.5-flash",
    description="Handles general knowledge and web search queries. Use this for anything NOT weather-related.",
    instruction=(
        "You are a search specialist. Use Google Search to find current, accurate "
        "information for the user's query. Summarize the results clearly and concisely."
    ),
    tools=[google_search],
)


# ── Root agent (coordinator) ─────────────────────────────────────────────────

root_agent = Agent(
    name="root_agent",
    model="gemini-2.5-flash",
    description="Coordinator agent that routes requests to the appropriate sub-agent.",
    instruction=(
        "You are a helpful coordinator. Analyze the user's request and delegate to the right agent:\n"
        "- For weather, forecast, temperature, or conditions questions → delegate to weather_agent\n"
        "- For general knowledge, news, facts, or web lookups → use the search_agent tool\n"
        "Always delegate — do not try to answer directly."
    ),
    tools=[agent_tool.AgentTool(agent=search_agent)],
    sub_agents=[weather_agent],
)

Overwriting multi_agent/agent.py


## Verify files

In [7]:
!ls -la multi_agent/

total 28
drwxr-xr-x 3 root root 4096 Mar  5 19:26 .
drwxr-xr-x 6 root root 4096 Mar  5 19:26 ..
-rw-r--r-- 1 root root 7393 Mar  5 19:43 agent.py
-rw-r--r-- 1 root root  175 Mar  5 19:43 .env
-rw-r--r-- 1 root root    1 Mar  5 19:43 __init__.py
drwxr-xr-x 2 root root 4096 Mar  5 19:34 __pycache__


## Test agents with InMemoryRunner

In [25]:
from google.adk.runners import InMemoryRunner
from google.genai import types

# Import agent module
import importlib.util
spec = importlib.util.spec_from_file_location("multi_agent", "multi_agent/agent.py")
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

runner = InMemoryRunner(agent=mod.root_agent, app_name="multi_agent")
print("Runner created")


async def run_query(query, user_id="test_user"):
    """Send a query to the root agent and print all events."""
    print(f"\n{'='*60}")
    print(f"QUERY: {query}")
    print(f"{'='*60}")

    session = await runner.session_service.create_session(
        app_name="multi_agent",
        user_id=user_id,
    )

    message = types.Content(
        role="user", parts=[types.Part(text=query)]
    )

    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=message,
    ):
        author = event.author or "unknown"

        if event.content and event.content.parts:
            for part in event.content.parts:
                if hasattr(part, "function_call") and part.function_call:
                    fc = part.function_call
                    print(f"  [{author}] tool call: {fc.name}({dict(fc.args) if fc.args else {}})")
                elif hasattr(part, "function_response") and part.function_response:
                    fr = part.function_response
                    resp_str = str(dict(fr.response) if fr.response else {})
                    print(f"  [{author}] tool result: {fr.name} -> {resp_str[:200]}")
                elif hasattr(part, "text") and part.text:
                    print(f"  [{author}]: {part.text[:300]}")

        if hasattr(event, "actions") and event.actions:
            if hasattr(event.actions, "transfer_to_agent") and event.actions.transfer_to_agent:
                print(f"  [{author}] >>> transferring to: {event.actions.transfer_to_agent}")

    print()

Runner created


### Test 1 — weather query (routes to weather_agent)

In [13]:
await run_query("What is the weather in Denver, Colorado?")


QUERY: What is the weather in Denver, Colorado?
  [root_agent] tool call: transfer_to_agent({'agent_name': 'weather_agent'})
  [root_agent] tool result: transfer_to_agent -> {'result': None}
  [root_agent] >>> transferring to: weather_agent
  [weather_agent] tool call: get_coordinates({'city': 'Denver, Colorado'})
  [weather_agent] tool result: get_coordinates -> {'latitude': 39.7392358, 'longitude': -104.990251, 'formatted_address': 'Denver, CO, USA'}
  [weather_agent] tool call: get_current_conditions({'latitude': 39.7392358, 'longitude': -104.990251})
  [weather_agent] tool result: get_current_conditions -> {'station': 'KBJC', 'description': 'Clear', 'temperature_f': 64.4, 'temperature_c': 18, 'humidity': None, 'wind_speed_mph': 41.1, 'wind_direction_degrees': 100}
  [weather_agent]: The current weather in Denver, Colorado is clear with a temperature of 64.4°F (18°C). The wind is blowing from 100 degrees at 41.1 mph.



### Test 2 — search query (routes to search_agent)

In [19]:
await run_query("Who won the last Super Bowl?")


QUERY: Who won the last Super Bowl?
  [root_agent] tool call: search_agent({'request': 'Who won the last Super Bowl?'})
  [root_agent] tool result: search_agent -> {'result': 'The Seattle Seahawks won the last Super Bowl, Super Bowl LX, defeating the New England Patriots on February 8, 2026.'}
  [root_agent]: The Seattle Seahawks won the last Super Bowl.



In [ ]:
# dataframe:
# uuid: 5A2BE9E8-F270-4148-B3DE-F64144E3F661
# output_variable:
# config_str:

import google.colabsqlviz.explore_dataframe as _vizcell
_vizcell.explore_dataframe(df_or_df_name='', uuid='5A2BE9E8-F270-4148-B3DE-F64144E3F661')

### Test 3 — weather query (different city)

In [ ]:
await run_query("What are the current conditions in Miami, Florida?")

### Test 4 — search query

In [15]:
await run_query("What is the Google Agent Development Kit?")


QUERY: What is the Google Agent Development Kit?
  [root_agent] tool call: search_agent({'request': 'What is the Google Agent Development Kit?'})
  [root_agent] tool result: search_agent -> {'result': "The Google Agent Development Kit (ADK) is an open-source, flexible, and modular framework designed to simplify the creation, deployment, and orchestration of AI agents and multi-agent syst
  [root_agent]: The Google Agent Development Kit (ADK) is an open-source framework that simplifies the creation, deployment, and orchestration of AI agents and multi-agent systems. It allows developers to build complex applications by composing multiple specialized agents, define flexible workflows, and equip agent



## Run via CLI

In [26]:
# Interactive CLI — type messages, Ctrl+C to stop
!adk run multi_agent

Log setup complete: /tmp/agents_log/agent.20260305_195111.log
To access latest log: tail -F /tmp/agents_log/agent.latest.log
/usr/local/lib/python3.12/dist-packages/google/adk/cli/cli.py:189: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  credential_service = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
Running agent root_agent, type exit to exit.
[user]: What is the weather in chicago
[weather_agent]: This afternoon in Chicago: Areas of fog and a slight chance of drizzle. Cloudy, with a high near 39 degrees Fahrenheit. North wind around 

In [ ]:
# Web UI — browser chat interface, Ctrl+C to stop
!adk web